# MMELON Model Fine-Tuning

Fine-tune the MMELON (Multi-Modal Encoder for Learning on Molecules) model on hit prediction data.

## Overview

This notebook demonstrates how to fine-tune MMELON for binary classification tasks in drug discovery:

**What is MMELON?**
- Multi-modal molecular encoder combining multiple molecular representations
- Integrates graph neural networks, fingerprints, and other molecular features
- Pre-trained on large-scale molecular datasets

**Workflow:**
1. **Setup & Configuration** - Environment setup and hyperparameter configuration
2. **Data Preparation** - Load screening data and create data loaders
3. **Model Initialization** - Load pre-trained MMELON and configure prediction head
4. **Training Setup** - Configure PyTorch Lightning trainer
5. **Fine-Tuning** - Train the model on your dataset
6. **Evaluation** - Assess model performance and save results

**Use Cases:**
- Hit prediction from screening campaigns
- Activity classification (active/inactive)
- Lead optimization prioritization

In [ ]:
def in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False
IN_COLAB = in_colab()
print(f"{IN_COLAB=}")

## Installation Instructions

### Local Environment
```bash
conda create -n mmelon311 python=3.11 -y
conda activate mmelon311
git clone https://github.com/jmorrone/biomed-multi-view.git
pip install git+https://github.com/jmorrone/biomed-multi-view.git
pip install --index-url https://download.pytorch.org/whl/cu121 torch==2.1.0 torchvision==0.16.0
pip install -f https://data.pyg.org/whl/torch-2.1.0+cu121.html "pyg_lib==0.4.0+pt21cu121" "torch_scatter==2.1.2+pt21cu121" "torch_cluster==1.6.3+pt21cu121" "torch_spline_conv==1.2.2+pt21cu121" --upgrade-strategy only-if-needed
pip install -q notebook ipykernel scikit-learn pandas openpyxl rdkit
```

In [ ]:
if IN_COLAB:
    print("***** Select runtime 2025.07!!!")
    !git clone https://github.com/jmorrone/biomed-multi-view.git

In [ ]:
if IN_COLAB:
    !pip install git+https://github.com/jmorrone/biomed-multi-view.git  

In [ ]:
if IN_COLAB:
    import os
    os.chdir('biomed-multi-view')
    !pip install -r requirements.txt

## 1. Setup and configuration

Edit these paths and hyperparameters according to your needs.

In [ ]:
import os
import sys

if sys.platform == "darwin":
    os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

MODEL_NAME = 'model_image' # 'model_graph', 'model_image', 'model_text', 'model_multiview'

NAME='molnet_bace_random'
DATA_PATH = "/Users/jamorron/Research/os_biomed_multiview/data/bace"
MODELS_DIR = "/Users/jamorron/Research/os_biomed_multiview/output"
MODEL_DIR = os.path.join(MODELS_DIR, NAME, MODEL_NAME)
DEMO_SIZE = None

# ============================================================================
# MODEL CONFIGURATION
# ============================================================================
BASE_MODEL = "ibm-research/biomed.sm.mv-te-84m"  # Hugging Face model name

# ============================================================================
# TRAINING HYPERPARAMETERS
# ============================================================================
EPOCHS = 1 # 10  # Number of training epochs
BATCH_SIZE = 16  # Training batch size
LEARNING_RATE = 2e-5  # Learning rate
WEIGHT_DECAY = 0.01  # Weight decay for regularization
SEED = 42  # Random seed for reproducibility

# ============================================================================
# DEVICE CONFIGURATION
# ============================================================================
DEVICE = None  # None for auto-detection, or specify 'cuda' or 'cpu'

print("Configuration loaded successfully!")
print(f"Output model directory: {MODEL_DIR}")
print(f"Base Model: {BASE_MODEL}")
print(f"Training epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")

## 2. Load and Prepare Data

Load the dataset splits and prepare them for training.

In [ ]:
import numpy as np
import pandas as pd
def reduce_data_for_demo(df: pd.DataFrame, demo_size: int, label_column: str, seed: int=42)-> pd.DataFrame:
          
    # Sample at most demo_size samples per class (label=0 and label=1)
    df_list = []
    for label in df[label_column].unique():
        df_label = df[df[label_column] == label]
        n = len(df_label)
        if n > demo_size:
            df_label = df_label.sample(n=demo_size, random_state=seed)
            print(f"Class {label}: {n} samples reduced to {len(df_label)} samples for demo.")
        else:
            print(f"Class {label}: {n} samples (no reduction needed for demo).")
        df_list.append(df_label)

    # Combine and shuffle
    df = pd.concat(df_list).sample(frac=1, random_state=seed).reset_index(drop=True)
    
    return df

In [ ]:
import pandas as pd
import numpy as np
# Load split data

split_path = DATA_PATH

print(f"Loading split from: {split_path}")

data_dir = DATA_PATH if DEMO_SIZE is None else DEMO_DATA_PATH
for split in ['train', 'val', 'test']:
    df = pd.read_csv(f"{split_path}/{split}.csv")
    
    if DEMO_SIZE is not None:
        original_size = len(df)
        df = reduce_data_for_demo(df, demo_size=DEMO_SIZE, label_column='label', seed=42)
        print(f"***** DEMO MODE: Reduced {split} set from {original_size} to {len(df)} samples (max {DEMO_SIZE} per class)")
    
    output_file = f"{data_dir}/data_{split}.csv"
    df.to_csv(output_file, index=False)
    print(f"Saved {output_file} for MMELON fine-tuning.")
    print(f"***  {split}: {len(df):,} samples ({df['label'].mean()*100:.2f}% positive)")

## 3. Prepare Data Loaders

In [ ]:
import pytorch_lightning as pl
from torch.utils.data import DataLoader

from bmfm_sm.core.data_modules.namespace import LateFusionStrategy, TaskType
from bmfm_sm.predictive.modules.finetune_lightning_module import FineTuneLightningModule
from bmfm_sm.predictive.data_modules.multimodal_finetune_dataset import MultiModalFinetuneDataPipeline
from bmfm_sm.predictive.data_modules.graph_finetune_dataset import Graph2dFinetuneDataPipeline
from bmfm_sm.predictive.data_modules.image_finetune_dataset import ImageFinetuneDataPipeline
from bmfm_sm.predictive.data_modules.text_finetune_dataset import TextFinetuneDataPipeline
from bmfm_sm.api.smmv_api import SmallMoleculeMultiViewModel

In [ ]:
# Set random seed for reproducibility
pl.seed_everything(SEED)

# Create output directory
os.makedirs(MODEL_DIR, exist_ok=True)

print("="*80)
print("INITIALIZING MMELON MODEL")
print("="*80)


In [ ]:

# Dataset arguments for MMELON framework

dataset_args = {
    'task_type': TaskType.CLASSIFICATION,
    'num_tasks': 1,  # Single task for binary classification
    'smiles_col': 'smiles',
    'label_cols': ['label'],
    'split_col': None
}

if MODEL_NAME=='model_multiview':
    dataset_args['modalities'] = ['TEXT_MODEL', 'IMAGE_MODEL', 'GRAPH_2D_MODEL']
    ModelFinetuneDataPipeline = MultiModalFinetuneDataPipeline
elif MODEL_NAME=='model_graph':
    ModelFinetuneDataPipeline = Graph2dFinetuneDataPipeline
elif MODEL_NAME=='model_image':
    ModelFinetuneDataPipeline = ImageFinetuneDataPipeline
elif MODEL_NAME=='model_text':
    ModelFinetuneDataPipeline = TextFinetuneDataPipeline


print(f"Creating data modules using data in {data_dir}...")
train_dataset = ModelFinetuneDataPipeline(
    data_dir=str(data_dir),
    dataset_args=dataset_args,
    stage='train'
)


val_dataset = ModelFinetuneDataPipeline(
    data_dir=str(data_dir),
    dataset_args=dataset_args,
    stage='val'
)


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=train_dataset.collate_fn,
    num_workers=4
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=val_dataset.collate_fn,
    num_workers=4
)


print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## 4. Initialize MMELON Model

In [ ]:
def get_singleview_state_dict(old_state_dict,model_name='model_graph'):
    new_state_dict = {}

    for key, value in old_state_dict.items():
        if model_name in key:
            new_key = key.replace(model_name+'.', '') # Apply your specific naming logic
            new_state_dict[new_key] = value

    # Now new_state_dict is ready to be loaded into the target model
    return new_state_dict

In [ ]:
FUSION_STRATEGY = LateFusionStrategy.ATTENTIONAL

BASE_MODEL = "ibm-research/biomed.sm.mv-te-84m"  # Hugging Face model name
EPOCHS = 1 # 10  # Number of training epochs
BATCH_SIZE = 16  # Training batch size
LEARNING_RATE = 2e-5  # Learning rate
WEIGHT_DECAY = 0.01  # Weight decay for regularization
SEED = 42  # Random seed for reproducibility


In [ ]:
if MODEL_NAME=='model_graph':
    base_model_class='bmfm_sm.predictive.modules.graph_2d_models.Graph2dModel'
    model_params = {
        'load_softmax': False,
        'encoder_embed_dim': 512,
        'num_tasks': 1
    }
elif MODEL_NAME=='model_image':
    base_model_class='bmfm_sm.predictive.modules.image_models.ImageModel'
    model_params = {
        'baseModel': "ResNet18"
    }
elif MODEL_NAME=='model_text':
    base_model_class='bmfm_sm.predictive.modules.text_models.TextModel'
    model_params = {
        'n_vocab': 2362,
        'n_embd': 768,
        'n_layer': 12,
        'n_head': 12,
        'num_feats': 32,
        'd_dropout': 0.2,
        'seed': 12345
    }
elif MODEL_NAME=='model_multiview':
    base_model_class='bmfm_sm.predictive.modules.smmv_model.SmallMoleculeMultiView'
    model_params = {
        'agg_arch': FUSION_STRATEGY.value[0],
        'agg_gate_input': FUSION_STRATEGY.value[1],
        'agg_weight_freeze': FUSION_STRATEGY.value[2],
        'inference_mode': False
    }


In [ ]:


# Finetuning arguments
finetuning_args = {
    'weight_freeze': 'unfrozen',
    'initialization': 'default',
    'head_arch': 'mlp',
    'use_norm': True,
    'head_dropout': 0.2
}

# Create Lightning module
print("Creating Lightning module...")
lightning_module = FineTuneLightningModule(
    base_model_class=base_model_class,
    model_params=model_params,
    task_type='classification',
    num_tasks=1,
    checkpoint_path=None,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    finetuning_args=finetuning_args
)

# Load pretrained weights from HuggingFace
print(f"Loading pretrained model from HuggingFace: {BASE_MODEL}")
pretrained_model = SmallMoleculeMultiViewModel.from_pretrained(
    FUSION_STRATEGY,
    model_path=BASE_MODEL,
    huggingface=True
)

if MODEL_NAME=='model_multiview':
    mismatched_keys = lightning_module.model.load_state_dict(pretrained_model.state_dict(), strict=False)
else:
    singleview_state_dict = get_singleview_state_dict(pretrained_model.state_dict(),model_name=MODEL_NAME)
    mismatched_keys = lightning_module.model.load_state_dict(singleview_state_dict, strict=False)

# You can also access the lists of keys directly
print("Missing keys:", mismatched_keys.missing_keys)
print("Unexpected keys:", mismatched_keys.unexpected_keys)

print("Model initialized successfully!")

## 5. Configure Trainer

In [ ]:
from pytorch_lightning.callbacks import ModelCheckpoint

# Setup callbacks
callbacks = [
    ModelCheckpoint(
        dirpath=MODEL_DIR,
        filename='mmelon-{epoch:02d}-{val_loss:.2f}',
        save_top_k=3,
        monitor='val_loss',
        mode='min',
        save_last=True,
        every_n_epochs=1
    )
]

# Create trainer
trainer = pl.Trainer(
    max_epochs=EPOCHS,
    callbacks=callbacks,
    default_root_dir=MODEL_DIR,
    accelerator='auto',
    devices=1,
    log_every_n_steps=10,
    val_check_interval=1.0,
    enable_checkpointing=True,
    enable_model_summary=True,
    precision=32
)


## 6. Fine-Tune Model

In [ ]:
%%time

print("="*80)
print("STARTING TRAINING")
print("="*80)

trainer.fit(lightning_module, train_loader, val_loader)

print("="*80)
print("TRAINING COMPLETE!")
print("="*80)

In [ ]:
os.listdir(MODEL_DIR)